# 12 — Fragility: controller retuned, control-sensor offset, monitoring drift, offsets beyond assumed (F2, F4, F5)
All tests use the production posterior (trained on the nominal controller and the assumed calibration sd).
* **Retuned controller**: Kp × {0.5, 2}, τI × {0.5, 2} in the plant only.
* **Control-sensor offset** oT ∈ {0.1, 0.25, 0.5, 1} K (in the loop: the plant moves, the reading stays at
  setpoint), and **monitoring drift** {0.5, 1, 2} K (outside the loop).
* **Calibration offsets beyond assumed**: every offset drawn at 2× and 3× its assumed sd.
* **Valve saturation** (decision 14; not a catalogue scenario): windows at α = 1.2, β = 0.41, valve at
  Qc_max. This sits at the prior's β edge, so coverage is reported with that caveat (System I nb29 lesson).
* **Residual detector** z_T = (mean logged T − Tsp)/(σ_T/√120): which of these it can see.

**PRE-REGISTER:** silent failure as in notebook 10; detector threshold fixed on a separate nominal set
(saturation guard as in System I).

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
PROD = json.loads((R / "06_production.json").read_text())
_m = [pickle.load(open(f, "rb")) for f in PROD["member_files"]]
post = inf.EnsemblePosterior(_m) if PROD["use_ensemble"] else _m[0]
print("production:", PROD["production_budget"], "ensemble" if PROD["use_ensemble"] else "single", PROD["feature_set"])

In [ ]:
def assess(label, **kw):
    sc = da.scenario_set(scn.GATED, 100, seed=12100, **kw)
    smp, leak = inf.sample_within_prior(post, sc["x"], 1000, rng_seed=0)
    _, r = wm.evaluate_scenario_set(smp, sc["theta"], sc["sc_name"], inf.LO, inf.HI)
    return [{"case": label, "leak": float(leak.mean()), **x} for x in r]
rows = assess("nominal")
for fk, fv in (("Kp", 0.5), ("Kp", 2.0), ("taui", 0.5), ("taui", 2.0)):
    rows += assess(f"{fk}x{fv}", plant=replace(P, **{fk: getattr(P, fk) * fv}))
for o in (0.1, 0.25, 0.5, 1.0):
    rows += assess(f"control_offset_{o}K", oT=o)
for dr in (0.5, 1.0, 2.0):
    rows += assess(f"monitor_drift_{dr}K", drift_T_monitor=dr)
sat = da.make_windows(np.tile([1.2, 0.41], (100, 1)), 12200)                 # valve saturation (decision 14)
smp, leak = inf.sample_within_prior(post, sat["x"], 1000, rng_seed=0)
_, r = wm.evaluate_scenario_set(smp, sat["theta"], np.array(["saturated_a1.2_b0.41"] * 100), inf.LO, inf.HI)
rows += [{"case": "valve_saturation", "leak": float(leak.mean()), **x} for x in r]
res = pd.DataFrame(rows); res.to_csv(R / "12_controller_and_sensors.csv", index=False)
display(res.pivot_table(index="case", columns="scenario", values="cov90_truth_b").round(2))

In [ ]:
import system_iii.sensors as S
rows = []
for mult in (2.0, 3.0):
    old = dict(S.CAL_SD)
    try:
        for k in S.CAL_SD:
            S.CAL_SD[k] = old[k] * mult
        rows += assess(f"offsets_{mult}x_assumed")
    finally:
        S.CAL_SD.update(old)
pd.DataFrame(rows).to_csv(R / "12_offsets_beyond_assumed.csv", index=False)
display(pd.DataFrame(rows).pivot_table(index="case", columns="scenario", values=["cov90_truth_b", "bias_b"]).round(3))

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).